In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

!ls /kaggle/input/q1-stage-3-2026

In [ ]:
# Needed to check for extensions common ones are jpg, jpeg, and png. I did manually check the dataset tbh but I thought I'd add this here! I do not remember how to increment per occurance but this does
# show me that there are no files in PNG or JPEG. Why not other extensions? Honestly, I manually inspected the files and thought hey, maybe you should use some bash.
!echo "PNGs"
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___healthy | grep 'PNG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Early_blight/ | grep 'PNG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Late_blight/ | grep 'PNG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___healthy | grep 'PNG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Early_blight/ | grep 'PNG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Late_blight/ | grep 'PNG'

!echo "JPEGs"
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___healthy | grep 'JPEG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Early_blight/ | grep 'JPEG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Late_blight/ | grep 'JPEG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___healthy | grep 'JPEG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Early_blight/ | grep 'JPEG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Late_blight/ | grep 'JPEG'
!echo "JPGs"
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___healthy | grep 'JPG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Early_blight/ | grep 'JPG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/test/Potato___Late_blight/ | grep 'JPG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___healthy | grep 'JPG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Early_blight/ | grep 'JPG'
!ls /kaggle/input/q1-stage-3-2026/PlantVillage/train/Potato___Late_blight/ | grep 'JPG'


In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

#No time to implement train dataset.

root = path
batch_size = 32
random.seed(42)


def find_class_root(start):
    for dirpath, dirnames, filenames in os.walk(start):
        dirnames = [d for d in dirnames if not d.startswith(".")] #Basically if it has an extension
        if len(dirnames) >= 3: #This is just so we skip the initial directories
            ok = 0
            for d in dirnames:
                folder = os.path.join(dirpath, d) #Concat
                try: #Try catch block in case I miss something. I would rather that it just skips any edge extension over me accounting for it
                    if any(f.endswith(".JPG") for f in os.listdir(folder)):
                        ok += 1 #Increment on finding a valid JPG file
                except:
                    pass
            if ok >= 2: #If there are 2 valid JPG files I can only assume we're working with a directory that holds data
                return dirpath
    return None

data_dir = find_class_root(root)
print("Using:", data_dir)

classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
print("Classes:", classes)



train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(15),
    transforms.Resize((32,32))
])

class PotatoDataset(Dataset):
    def __init__(self, root, classes, transform = None):
        self.samples = []
        self.classes = classes
        self.transform = transform
        self.map = {c:i for i,c in enumerate(classes)}
        for c in classes:
            folder = os.path.join(root, c)
            for f in os.listdir(folder):
                if f.endswith('.JPG'):
                    self.samples.append((os.path.join(folder, f), self.map[c]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image, label = self.samples[idx]
        img = Image.open(image).convert("RGB")
        img =self.transform(img)
        return img, label

train_dataset = PotatoDataset(data_dir, classes ,transform=train_transform)
print("Total images:")




train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

imgs, label = next(iter(train_loader))

plt.figure(figsize=(10,6))
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.imshow(imgs[i].permute(1,2,0))
    plt.title(classes[label[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Write your code here
import torch
import torch.nn as nn
import torch.nn.functional as F

class PotatoCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn4   = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn5   = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(256*1*1, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32→16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16→8
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 8→4
        x = self.pool(F.relu(self.bn4(self.conv4(x))))  # 4→2
        x = self.pool(F.relu(self.bn5(self.conv5(x))))  # 2→1

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model = PotatoCNN()
print(model)


In [ ]:
# Write your code here
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def validate(model, loader, criterion, device):
    model.eval()
    with torch.no_grad():
      total_loss = 0.0
      correct = 0
      total = 0

      for images, labels in loader:
          images = images.to(device)
          labels = labels.to(device)

          outputs = model(images)
          loss = criterion(outputs, labels)

          total_loss += loss.item() * labels.size(0)
          preds = outputs.argmax(dim=1)
          correct += (preds == labels).sum().item()
          total += labels.size(0)

      return total_loss / total, correct / total


In [ ]:
# Write your code here
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 10

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)


    train_losses.append(tr_loss)
    train_accs.append(tr_acc)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} ")

plt.figure(figsize=(8,8))
plt.plot(train_losses, label="Train Loss")
plt.legend()
plt.title("Loss")
plt.show()

plt.figure(figsize=(8,8))
plt.plot(train_accs, label="Train Acc")
plt.legend()
plt.title("Accuracy")
plt.show()

In [ ]:
#Just gonna skip this one. 2 minutes on the clock